# Vimeo folder to Course ouline

## Get folder items

Sử dụng Vimeo API để lấy danh sách video ở một folder trên Vimeo

In [ ]:
const options = {
    method: "GET",
    redirect: "follow",
    headers: {
        "Authorization": `Bearer ${accessToken}`,
        "Content-Type": "application/json"
    }
};
// Make a GET request to the Vimeo API
apiUrl = `https://api.vimeo.com/users/${myID}/projects/${projectID}/items`
const response = UrlFetchApp.fetch(apiUrl, options)
data = JSON.parse(response.getContentText())

## Get videos with infomation

Từ list item lấy danh sách video

In [ ]:
const videos = [];
for (const item of data.data) {
    const name = parse_name(item.video.name)
    const pk = name[0]
    const title = name[1]
    const video = {
        pk,
        url: item.video.link,
        title,
        description: item.video.description,
        duration: get_duration(item.video.duration)
    }
    videos.push(video)
}

Với tiêu đề video lấy số thự tự của video

In [ ]:
function parse_name(str) {
    const regex = /((Bài|Buổi)\s)?(\d+)(( - )|(: ))(.*)/;
    const match = str.match(regex);
    if (match) {
        const extractedNumber = parseInt(match[3], 10);
        const extractedString = match[6];
        return [extractedNumber, extractedString]
    } else return ["", str];
}

Từ output sau khi GET API sẽ phân ra từng section dựa trên Description của Video

In [58]:
% % script node

data = [{
    "pk": "",
    "url": "https://vimeo.com/821909358",
    "title": "Weekly LLD Meeting 2023-04-28 02:42:37",
    "description": null,
    "duration": "55:15"
}, {
    "pk": "",
    "url": "https://vimeo.com/821846221",
    "title": "Weekly LLD Meeting 2023-04-27 23:12:02",
    "description": "A - Laladay MEET",
    "duration": "10:30"
}, {
    "pk": "",
    "url": "https://vimeo.com/820755680",
    "title": "Weekly LLD Meeting 2023-04-25 04:01:28",
    "description": null,
    "duration": "13:02"
}, {
    "pk": "",
    "url": "https://vimeo.com/810021916",
    "title": "Weekly LLD Meeting 2023-03-21 03:25:21",
    "description": "B",
    "duration": "30:12"
}]


function videos2outline(videos) {
    const groupedVideos = videos.reduce((acc, video) => {
        const description = video.description
        if (!acc[description]) {
            acc[description] = [];
        }
        acc[description].push({
            title: video.title,
            url: video.url,
            duration: video.duration,
        });
        return acc;
    }, {});

    const sections = Object.keys(groupedVideos).map((description) => ({
        heading: description.split(" - ")[0] || 'General',
        description: description.split(" - ")[1] || '', // You can customize this if needed
        lessons: groupedVideos[description],
    }));

    return JSON.stringify(sections, null, 2);
}
HTML = `document.currentScript.setAttribute("outline", JSON.stringify(` + videos2outline(data) + `))`;
console.log(HTML);

document.currentScript.setAttribute("outline", JSON.stringify([
  {
    "heading": "null",
    "description": "",
    "lessons": [
      {
        "title": "Weekly LLD Meeting 2023-04-28 02:42:37",
        "url": "https://vimeo.com/821909358",
        "duration": "55:15"
      },
      {
        "title": "Weekly LLD Meeting 2023-04-25 04:01:28",
        "url": "https://vimeo.com/820755680",
        "duration": "13:02"
      }
    ]
  },
  {
    "heading": "A",
    "description": "Laladay MEET",
    "lessons": [
      {
        "title": "Weekly LLD Meeting 2023-04-27 23:12:02",
        "url": "https://vimeo.com/821846221",
        "duration": "10:30"
      }
    ]
  },
  {
    "heading": "B",
    "description": "",
    "lessons": [
      {
        "title": "Weekly LLD Meeting 2023-03-21 03:25:21",
        "url": "https://vimeo.com/810021916",
        "duration": "30:12"
      }
    ]
  }
]))
